# 04 – Comparing the groups

**Input:** `results/peptide_features.csv` (from notebook 03)  
**Outputs:** figures in `figures/` and tables `results/comparison_*.csv`

All comparisons use the **non-redundant set**: only the CD-HIT representatives (90% identity, notebook 02). Section 1c shows how much this changes the answer.

We answer the three questions from the study plan:

* **Q1** – How do *Natural* and *Synthetic* peptides differ in length, charge and amino-acid makeup?
* **Q2** – How do peptides that target different organisms (Gram+, Gram-, fungi, cancer, virus) differ?
* **Q3** – Do peptides active against *mammalian cells* (i.e. cytotoxic) look different from the others?

**Method in one paragraph.** The features are not normally distributed, so we use rank-based tests: **Mann-Whitney U** for two groups and **Kruskal-Wallis** for more than two. With thousands of peptides almost every difference is "significant", so the p-value alone says little. We therefore always report an **effect size**, Cliff's delta (δ): the probability that a random peptide from group A has a higher value than one from group B, minus the reverse. δ = 0 means no difference; δ = ±1 means the groups do not overlap at all.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations
from scipy.stats import mannwhitneyu, kruskal

allp = pd.read_csv("../results/peptide_features.csv")     # every cleaned peptide (before CD-HIT)
df = allp[allp["is_representative"]].copy()              # non-redundant set used for all analyses
FIG, RES = "../figures", "../results"

FEATURES = ["length", "net_charge", "charge_density", "hydrophobic_fraction"]
AA = list("ACDEFGHIKLMNPQRSTVWY")
AA_COLS = [f"pct_{a}" for a in AA]
COLORS = {"Natural": "#2a9d8f", "Synthetic": "#e76f51"}
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print(f"{len(allp)} cleaned peptides -> {len(df)} non-redundant representatives")

18196 cleaned peptides -> 9622 non-redundant representatives


## Helper functions
Small and reusable: one test function, one effect-size label.

In [2]:
def cliffs_delta(a, b):
    """P(a > b) - P(a < b), computed from the Mann-Whitney U statistic."""
    u, p = mannwhitneyu(a, b, alternative="two-sided")
    return 2 * u / (len(a) * len(b)) - 1, p

def size_label(d):
    d = abs(d)
    return "negligible" if d < 0.147 else "small" if d < 0.33 else "medium" if d < 0.474 else "large"

def compare_two(data, group_col, g1, g2, features):
    """Mann-Whitney + Cliff's delta for each feature, g1 vs g2."""
    a, b = data[data[group_col] == g1], data[data[group_col] == g2]
    rows = []
    for f in features:
        d, p = cliffs_delta(a[f], b[f])
        rows.append({"feature": f, f"median_{g1}": a[f].median(), f"median_{g2}": b[f].median(),
                     "cliffs_delta": round(d, 3), "effect": size_label(d), "p_value": p})
    return pd.DataFrame(rows)

# tiny self-test: fully separated groups -> delta = +1 / -1, identical groups -> 0
assert cliffs_delta(pd.Series([5, 6, 7]), pd.Series([1, 2, 3]))[0] == 1
assert cliffs_delta(pd.Series([1, 2, 3]), pd.Series([5, 6, 7]))[0] == -1
assert abs(cliffs_delta(pd.Series([1, 2, 3]), pd.Series([1, 2, 3]))[0]) < 1e-9
print("helpers OK")

helpers OK


# Q1 – Natural vs Synthetic
## 1a. Sample sizes (non-redundant set)

In [3]:
df["origin"].value_counts()

origin
Synthetic    7314
Natural      2308
Name: count, dtype: int64

## 1b. Length, charge and hydrophobicity

In [4]:
q1 = compare_two(df, "origin", "Natural", "Synthetic", FEATURES)
q1.to_csv(f"{RES}/comparison_q1_origin.csv", index=False)
q1.round({"median_Natural": 2, "median_Synthetic": 2}).style.format({"p_value": "{:.1e}"})

,feature,median_Natural,median_Synthetic,cliffs_delta,effect,p_value
0,length,25.000000,15.000000,0.467000,medium,2.5e-252
1,net_charge,3.000000,4.000000,-0.258000,small,2.7e-79
2,charge_density,0.100000,0.290000,-0.598000,large,0.0e+00
3,hydrophobic_fraction,0.410000,0.450000,-0.164000,small,1.6e-32


In [5]:
titles = {"length": "Length (residues)", "net_charge": "Net charge",
          "charge_density": "Charge per residue", "hydrophobic_fraction": "Hydrophobic fraction"}
fig, axes = plt.subplots(1, 4, figsize=(12, 3.6))
for ax, f in zip(axes, FEATURES):
    data = [df.loc[df.origin == o, f] for o in ["Natural", "Synthetic"]]
    bp = ax.boxplot(data, tick_labels=["Natural", "Synthetic"], showfliers=False, patch_artist=True,
                    medianprops={"color": "black"})
    for patch, o in zip(bp["boxes"], ["Natural", "Synthetic"]):
        patch.set_facecolor(COLORS[o]); patch.set_alpha(0.8)
    d = q1.set_index("feature").loc[f]
    ax.set_title(f"{titles[f]}\nδ = {d.cliffs_delta:+.2f} ({d.effect})", fontsize=10)
fig.suptitle("Natural vs Synthetic peptides (outliers hidden)", y=1.02)
plt.tight_layout()
plt.savefig(f"{FIG}/q1_boxplots.png", dpi=150, bbox_inches="tight")
plt.show()

## 1c. Does removing redundancy change the answer?
The same comparison on all 18,196 cleaned peptides (before CD-HIT) and on the non-redundant representatives.

In [6]:
rows = []
for label, data in [("before CD-HIT", allp), ("after CD-HIT", df)]:
    r = compare_two(data, "origin", "Natural", "Synthetic", FEATURES)
    r.insert(0, "dataset", label)
    r.insert(1, "n_nat/n_syn", f"{(data.origin=='Natural').sum()}/{(data.origin=='Synthetic').sum()}")
    rows.append(r)
red = pd.concat(rows, ignore_index=True)
red.to_csv(f"{RES}/comparison_q1_redundancy.csv", index=False)
red.pivot(index="feature", columns="dataset", values="cliffs_delta")[["before CD-HIT", "after CD-HIT"]].loc[FEATURES]

dataset,before CD-HIT,after CD-HIT
feature,,
length,0.441,0.467
net_charge,-0.280,-0.258
charge_density,-0.598,-0.598
hydrophobic_fraction,-0.079,-0.164


## 1d. Robustness: is it just because natural peptides are longer?
Natural peptides are longer, and length affects the other features. To check, repeat the comparison using only **Short (10–24 aa)** peptides, and separately only peptides **without D-amino acids** (which are much more common in synthetic peptides).

In [7]:
subsets = {
    "all peptides": df,
    "Short (10-24 aa) only": df[df.length_class == "Short (10–24 aa)"],
    "no D-amino acids": df[~df.has_D_aa],
}
rows = []
for name, sub in subsets.items():
    res = compare_two(sub, "origin", "Natural", "Synthetic", ["length", "charge_density", "hydrophobic_fraction"])
    res.insert(0, "subset", name)
    res.insert(1, "n_nat/n_syn", f"{(sub.origin=='Natural').sum()}/{(sub.origin=='Synthetic').sum()}")
    rows.append(res)
robust = pd.concat(rows, ignore_index=True)
robust.to_csv(f"{RES}/comparison_q1_robustness.csv", index=False)
robust.pivot(index="feature", columns="subset", values="cliffs_delta")[list(subsets)]

subset,all peptides,Short (10-24 aa) only,no D-amino acids
feature,,,
charge_density,-0.598,-0.662,-0.588
hydrophobic_fraction,-0.164,0.135,-0.171
length,0.467,0.279,0.492


## Amino-acid composition
The amino-acid comparison (with a background population and enrichment statistics) is in notebook **05_enrichment**.

## 1e. Which targets do the two groups hit?
Percentage of peptides in each origin that are active against each target (targets overlap, so columns don't add to 100%). Only peptides with target information are used.

In [8]:
target_cols = ["is_gram_pos", "is_gram_neg", "is_fungus", "is_cancer", "is_virus",
               "is_biofilm", "is_parasite", "is_mammalian_cell"]
t = df[df.has_target_info]
share = (t.groupby("origin")[target_cols].mean() * 100).T
share.index = [i.replace("is_", "") for i in share.index]
share["n_natural"] = t.groupby("origin")[target_cols].sum().T.loc[:, "Natural"].values
share["n_synthetic"] = t.groupby("origin")[target_cols].sum().T.loc[:, "Synthetic"].values
share.to_csv(f"{RES}/comparison_q1_targets.csv")

fig, ax = plt.subplots(figsize=(6.5, 3.6))
im = ax.imshow(share[["Natural", "Synthetic"]].values, cmap="Blues", vmin=0, vmax=100, aspect="auto")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Natural", "Synthetic"])
ax.set_yticks(range(len(share))); ax.set_yticklabels(share.index)
for i in range(len(share)):
    for j, o in enumerate(["Natural", "Synthetic"]):
        v = share.loc[share.index[i], o]
        ax.text(j, i, f"{v:.0f}%", ha="center", va="center", color="white" if v > 55 else "black")
ax.set_title("% of peptides active against each target", fontsize=10)
plt.tight_layout()
plt.savefig(f"{FIG}/q1_target_heatmap.png", dpi=150)
plt.show()
share.round(1)

origin,Natural,Synthetic,n_natural,n_synthetic
gram_pos,86.8,77.7,1990,5613
gram_neg,84.4,80.8,1936,5843
fungus,54.2,22.4,1242,1617
cancer,20.3,18.6,466,1347
virus,4.5,7.6,104,550
biofilm,2.2,3.9,51,282
parasite,5.0,0.9,114,67
mammalian_cell,52.1,57.2,1195,4132


# Q2 – Peptides with different targets
Target groups overlap (a peptide can hit Gram+, Gram- and fungi at once), so tests that treat them as separate samples would be invalid. To keep the groups **independent**, we only use peptides whose target list is *exactly* one of these:

| Group | Meaning |
|---|---|
| Gram+ only | only Gram-positive bacteria |
| Gram- only | only Gram-negative bacteria |
| Gram+ & Gram- | both bacterial types, nothing else |
| Fungus only | only fungi |
| Virus only | only viruses |
| Cancer only | only cancer cells |

In [9]:
active = [c for c in df.columns if c.startswith("is_") and c != "is_representative"]
t = df[df.has_target_info].copy()
n_targets = t[active].sum(axis=1)

def exclusive_group(r):
    on = frozenset(c for c in active if r[c] == 1)
    return {frozenset(["is_gram_pos"]): "Gram+ only", frozenset(["is_gram_neg"]): "Gram- only",
            frozenset(["is_gram_pos", "is_gram_neg"]): "Gram+ & Gram-",
            frozenset(["is_fungus"]): "Fungus only", frozenset(["is_virus"]): "Virus only",
            frozenset(["is_cancer"]): "Cancer only"}.get(on)

t["target_group"] = t.apply(exclusive_group, axis=1)
t = t.dropna(subset=["target_group"])
GROUPS = ["Gram+ only", "Gram- only", "Gram+ & Gram-", "Fungus only", "Virus only", "Cancer only"]
print(t["target_group"].value_counts().reindex(GROUPS))
print(f"\nOrigin mix inside each group (% Natural):")
print((t.groupby("target_group")["origin"].apply(lambda s: (s == "Natural").mean() * 100)).reindex(GROUPS).round(1))

target_group
Gram+ only        412
Gram- only        294
Gram+ & Gram-    1598
Fungus only       178
Virus only        363
Cancer only       121
Name: count, dtype: int64

Origin mix inside each group (% Natural):
target_group
Gram+ only       18.4
Gram- only       13.6
Gram+ & Gram-    22.2
Fungus only      33.7
Virus only        1.4
Cancer only      52.1
Name: origin, dtype: float64


## 2a. Overall difference between the groups (Kruskal-Wallis)

In [10]:
rows = []
for f in FEATURES:
    h, p = kruskal(*[t.loc[t.target_group == g, f] for g in GROUPS])
    # epsilon-squared: a simple effect size for Kruskal-Wallis (0 = no difference, 1 = complete)
    eps2 = (h - len(GROUPS) + 1) / (len(t) - len(GROUPS))
    rows.append({"feature": f, "H": round(h, 1), "epsilon_sq": round(eps2, 3), "p_value": p,
                 **{f"median_{g}": round(t.loc[t.target_group == g, f].median(), 2) for g in GROUPS}})
q2 = pd.DataFrame(rows)
q2.to_csv(f"{RES}/comparison_q2_targets.csv", index=False)
q2.style.format({"p_value": "{:.1e}"})

,feature,H,epsilon_sq,p_value,median_Gram+ only,median_Gram- only,median_Gram+ & Gram-,median_Fungus only,median_Virus only,median_Cancer only
0,length,156.800000,0.051000,4.7e-32,13.000000,17.000000,18.000000,12.000000,18.000000,8.000000
1,net_charge,514.400000,0.172000,6.4e-109,3.000000,5.000000,4.000000,3.000000,0.000000,0.000000
2,charge_density,498.100000,0.167000,2.1e-105,0.210000,0.270000,0.190000,0.170000,0.000000,0.000000
3,hydrophobic_fraction,154.600000,0.051000,1.4e-31,0.500000,0.360000,0.430000,0.370000,0.420000,0.500000


In [11]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, f in zip(axes, ["length", "charge_density", "hydrophobic_fraction"]):
    data = [t.loc[t.target_group == g, f] for g in GROUPS]
    bp = ax.boxplot(data, tick_labels=GROUPS, showfliers=False, patch_artist=True, medianprops={"color": "black"})
    for patch in bp["boxes"]:
        patch.set_facecolor("#8ecae6")
    ax.set_title(titles[f], fontsize=10)
    ax.tick_params(axis="x", rotation=40)
    for lab in ax.get_xticklabels():
        lab.set_ha("right")
plt.tight_layout()
plt.savefig(f"{FIG}/q2_boxplots.png", dpi=150, bbox_inches="tight")
plt.show()

## 2b. Which pairs of groups differ?
Pairwise Cliff's delta (row group vs column group, positive = row group has higher values). Each cell is a Mann-Whitney comparison; with 15 pairs per feature, only differences that are also at least "small" are worth discussing.

In [12]:
pair_feats = ["length", "charge_density", "hydrophobic_fraction"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), gridspec_kw={"wspace": 0.08})
pair_rows = []
for ax, f in zip(axes, pair_feats):
    M = np.zeros((len(GROUPS), len(GROUPS)))
    for i, j in combinations(range(len(GROUPS)), 2):
        d, p = cliffs_delta(t.loc[t.target_group == GROUPS[i], f], t.loc[t.target_group == GROUPS[j], f])
        M[i, j], M[j, i] = d, -d
        pair_rows.append({"feature": f, "group_1": GROUPS[i], "group_2": GROUPS[j],
                          "cliffs_delta": round(d, 3), "effect": size_label(d), "p_bonferroni": min(p * 15, 1)})
    im = ax.imshow(M, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(len(GROUPS))); ax.set_xticklabels(GROUPS, rotation=40, ha="right", fontsize=8)
    ax.set_yticks(range(len(GROUPS)))
    ax.set_yticklabels(GROUPS if ax is axes[0] else [], fontsize=8)   # labels only on the first panel
    for i in range(len(GROUPS)):
        for j in range(len(GROUPS)):
            if i != j:
                ax.text(j, i, f"{M[i, j]:+.2f}", ha="center", va="center", fontsize=7,
                        color="white" if abs(M[i, j]) > 0.6 else "black")
    ax.set_title(titles[f], fontsize=10)
fig.colorbar(im, ax=axes, shrink=0.8, label="Cliff's delta (row vs column)")
plt.savefig(f"{FIG}/q2_pairwise_delta.png", dpi=150, bbox_inches="tight")
plt.show()
pairs = pd.DataFrame(pair_rows)
pairs.to_csv(f"{RES}/comparison_q2_pairwise.csv", index=False)
print("pairs with at least a medium effect:")
pairs[pairs.cliffs_delta.abs() >= 0.33].sort_values("cliffs_delta", key=abs, ascending=False).head(12)

pairs with at least a medium effect:


,feature,group_1,group_2,cliffs_delta,effect,p_bonferroni
23,charge_density,Gram- only,Cancer only,0.773,large,2.490437e-34
22,charge_density,Gram- only,Virus only,0.756,large,2.587291e-61
26,charge_density,Gram+ & Gram-,Cancer only,0.591,large,2.726202e-26
25,charge_density,Gram+ & Gram-,Virus only,0.577,large,5.556226e-65
19,charge_density,Gram+ only,Cancer only,0.573,large,7.065174e-21
18,charge_density,Gram+ only,Virus only,0.571,large,7.115944e-42
28,charge_density,Fungus only,Cancer only,0.564,large,5.928824e-16
27,charge_density,Fungus only,Virus only,0.562,large,2.607783e-25
11,length,Gram+ & Gram-,Cancer only,0.535,large,1.105035e-21
14,length,Virus only,Cancer only,0.488,large,1.244338e-14


## 2c. Check: is this just Natural vs Synthetic again?
The origin mix differs a lot between the groups (see above: Virus-only peptides are almost all synthetic, Cancer-only about half natural). Since origin has a large effect on charge density (Q1), the Q2 differences could partly be an origin effect. To check, we repeat the medians using **Synthetic peptides only**.

In [13]:
syn = t[t.origin == "Synthetic"]
chk = syn.groupby("target_group")[["length", "charge_density", "hydrophobic_fraction"]].median().reindex(GROUPS)
chk.insert(0, "n", syn.groupby("target_group").size().reindex(GROUPS))
chk.round(2).to_csv(f"{RES}/comparison_q2_synthetic_only.csv")
chk.round(2)

,n,length,charge_density,hydrophobic_fraction
target_group,,,,
Gram+ only,336,12.0,0.27,0.50
Gram- only,254,15.0,0.33,0.38
Gram+ & Gram-,1244,17.0,0.24,0.44
Fungus only,118,9.0,0.33,0.44
Virus only,358,18.0,0.00,0.43
Cancer only,58,12.0,0.07,0.57


# Q3 – Mammalian-cell activity
`is_mammalian_cell = 1` means DBAASP lists mammalian cells among the peptide's targets, i.e. the peptide was found to be active on (cytotoxic/hemolytic to) mammalian cells. **Caveat:** `0` means "not reported", which is not the same as "proven safe" – the peptide may simply not have been tested on mammalian cells. Only peptides with target information are used.

In [14]:
m = df[df.has_target_info].copy()
m["mammalian"] = np.where(m.is_mammalian_cell == 1, "Mammalian-active", "Not reported")
print(m["mammalian"].value_counts(), "\n")

def q3_table(data, label):
    r = compare_two(data, "mammalian", "Mammalian-active", "Not reported", FEATURES)
    r.insert(0, "subset", label)
    return r

q3 = pd.concat([q3_table(m, "all"),
                q3_table(m[m.origin == "Synthetic"], "Synthetic only"),
                q3_table(m[m.origin == "Natural"], "Natural only")], ignore_index=True)
q3.to_csv(f"{RES}/comparison_q3_mammalian.csv", index=False)
q3.round(2).style.format({"p_value": "{:.1e}"})

mammalian
Mammalian-active    5327
Not reported        4194
Name: count, dtype: int64 



,subset,feature,median_Mammalian-active,median_Not reported,cliffs_delta,effect,p_value
0,all,length,16.000000,17.000000,-0.030000,negligible,1.0e-02
1,all,net_charge,4.000000,3.000000,0.180000,small,0.0e+00
2,all,charge_density,0.250000,0.170000,0.220000,small,0.0e+00
3,all,hydrophobic_fraction,0.460000,0.430000,0.140000,negligible,0.0e+00
4,Synthetic only,length,15.000000,15.000000,0.010000,negligible,2.9e-01
5,Synthetic only,net_charge,4.000000,3.000000,0.220000,small,0.0e+00
6,Synthetic only,charge_density,0.310000,0.230000,0.210000,small,0.0e+00
7,Synthetic only,hydrophobic_fraction,0.470000,0.440000,0.110000,negligible,0.0e+00
8,Natural only,length,24.000000,30.000000,-0.180000,small,0.0e+00
9,Natural only,net_charge,3.000000,3.000000,0.060000,negligible,2.0e-02


In [15]:
fig, axes = plt.subplots(1, 4, figsize=(12, 3.6))
for ax, f in zip(axes, FEATURES):
    data = [m.loc[m.mammalian == g, f] for g in ["Mammalian-active", "Not reported"]]
    bp = ax.boxplot(data, tick_labels=["Mammalian-\nactive", "Not\nreported"], showfliers=False,
                    patch_artist=True, medianprops={"color": "black"})
    for patch, c in zip(bp["boxes"], ["#c1121f", "#adb5bd"]):
        patch.set_facecolor(c); patch.set_alpha(0.8)
    d = q3[(q3.subset == "all") & (q3.feature == f)].iloc[0]
    ax.set_title(f"{titles[f]}\nδ = {d.cliffs_delta:+.2f} ({d.effect})", fontsize=10)
plt.tight_layout()
plt.savefig(f"{FIG}/q3_boxplots.png", dpi=150, bbox_inches="tight")
plt.show()

# Bonus – do the groups separate visually? (PCA)
A quick principal-component analysis on length, charge, charge density, hydrophobic fraction and the 20 composition percentages (all standardised). If Natural and Synthetic overlap heavily, they are not cleanly separable by these simple features.

In [16]:
X = df[FEATURES + AA_COLS].to_numpy(dtype=float)
X = (X - X.mean(axis=0)) / X.std(axis=0)
U, S, Vt = np.linalg.svd(X, full_matrices=False)
pcs = U[:, :2] * S[:2]
var = (S ** 2 / (S ** 2).sum())[:2] * 100

fig, ax = plt.subplots(figsize=(6, 5))
for o in ["Synthetic", "Natural"]:
    mask = (df.origin == o).to_numpy()
    ax.scatter(pcs[mask, 0], pcs[mask, 1], s=4, alpha=0.25, color=COLORS[o], label=o, rasterized=True)
ax.set_xlabel(f"PC1 ({var[0]:.0f}% of variance)"); ax.set_ylabel(f"PC2 ({var[1]:.0f}% of variance)")
ax.legend(markerscale=4, frameon=False)
ax.set_title("PCA of peptide features", fontsize=10)
plt.tight_layout()
plt.savefig(f"{FIG}/pca_origin.png", dpi=150)
plt.show()

top = pd.Series(Vt[0], index=FEATURES + AA_COLS).abs().sort_values(ascending=False).head(5)
print("features driving PC1:", ", ".join(top.index))

features driving PC1: charge_density, net_charge, pct_K, pct_R, pct_D


## One-page summary table

In [17]:
summary = pd.DataFrame([
    {"question": "Q1 Natural vs Synthetic", "feature": r.feature, "effect_size_delta": r.cliffs_delta, "effect": r.effect}
    for r in q1.itertuples()
] + [
    {"question": "Q3 Mammalian-active vs not", "feature": r.feature, "effect_size_delta": r.cliffs_delta, "effect": r.effect}
    for r in q3[q3.subset == "all"].itertuples()
])
summary.to_csv(f"{RES}/comparison_summary.csv", index=False)
summary

,question,feature,effect_size_delta,effect
0,Q1 Natural vs Synthetic,length,0.467,medium
1,Q1 Natural vs Synthetic,net_charge,-0.258,small
2,Q1 Natural vs Synthetic,charge_density,-0.598,large
3,Q1 Natural vs Synthetic,hydrophobic_fraction,-0.164,small
4,Q3 Mammalian-active vs not,length,-0.029,negligible
5,Q3 Mammalian-active vs not,net_charge,0.183,small
6,Q3 Mammalian-active vs not,charge_density,0.220,small
7,Q3 Mammalian-active vs not,hydrophobic_fraction,0.145,negligible


# Findings and limitations

All results use the **non-redundant set** (CD-HIT, 90% identity): 2,308 natural and 7,314 synthetic peptides. Numbers are medians unless stated; δ is Cliff's delta (|δ| < 0.15 negligible, < 0.33 small, < 0.47 medium, otherwise large). Everything is "statistically significant" at this sample size, so the discussion relies on effect sizes.

## Q1 – Natural vs Synthetic
* **Charge per residue is the clearest difference:** 0.10 (natural) vs 0.29 (synthetic), δ = −0.60 (large). It holds when only short (10–24 aa) peptides are compared (δ = −0.66) and when peptides with D-amino acids are removed (δ = −0.59), and it is exactly the same before and after redundancy removal (δ = −0.60).
* **Length:** natural peptides are longer, 25 vs 15 residues (δ = +0.47, medium). Raw net charge differs less (3 vs 4, δ = −0.26, small) because natural peptides are longer.
* **Hydrophobic fraction is a small difference that depends on length.** Overall synthetic peptides are slightly more hydrophobic (0.46 vs 0.41, δ = −0.16, small; before redundancy removal it looked negligible, δ = −0.08). Among short peptides the direction flips (natural 0.52 vs synthetic 0.46, δ = +0.14, negligible), so report it with care.
* **Effect of CD-HIT:** it kept 71% of natural but only 49% of synthetic peptides (synthetic entries are far more redundant). Charge density is unchanged, length and net charge moved by less than 0.03 in δ, and only the hydrophobic-fraction difference changed noticeably.
* **Targets:** natural peptides are reported active against fungi far more often (54% vs 22%) and against parasites (5.0% vs 0.9%); Gram+ activity is 87% vs 78%. Virus activity is more common among synthetic peptides (7.6% vs 4.5%).
* **PCA:** the two groups overlap heavily (PC1 and PC2 explain only 14% and 10% of the variance). Synthetic peptides sit further along PC1, which is driven mainly by charge, but the features do not separate the groups cleanly.
* Amino-acid composition is analysed against a background population in notebook 05.

## Q2 – Different targets (exclusive groups only)
Group sizes: Gram+ only 412, Gram- only 294, Gram+ & Gram- 1,598, Fungus only 178, Virus only 363, Cancer only 121.

* Groups differ mainly in **charge**: Kruskal-Wallis effect size ε² ≈ 0.17 for net charge and charge density, but only 0.05 for length and for hydrophobic fraction.
* **Virus-only** and **Cancer-only** peptides have median net charge 0, compared with 3–5 for the antibacterial and antifungal groups (charge-density δ between +0.56 and +0.77 in favour of the antibacterial and antifungal groups).
* **Gram- only** peptides have the highest charge density (0.27) and the lowest hydrophobic fraction (0.36, similar to Fungus-only at 0.37); Gram+ only peptides are more hydrophobic (0.50; δ = +0.40 against Gram- only).
* **Cancer-only** peptides are the shortest (median 8) and among the most hydrophobic (0.50), but the group is small.
* **Check:** the origin mix differs by group (Virus-only is 1% natural, Cancer-only 52%). Repeating the medians for synthetic peptides only gives the same picture (Virus-only charge density 0.00, Cancer-only 0.07, Gram- only 0.33), so this is not only an origin effect. The synthetic Cancer-only group is small (n = 58).

## Q3 – Mammalian-cell activity (5,327 mammalian-active vs 4,194 not reported)
* Peptides reported active on mammalian cells have somewhat higher charge density (0.25 vs 0.17, δ = +0.22) and net charge (4 vs 3, δ = +0.18): small effects. Hydrophobic fraction (0.46 vs 0.43, δ = +0.14) and length (δ = −0.03) are negligible.
* The charge effects appear within synthetic peptides (δ = +0.21 to +0.22) and within natural peptides (charge density δ = +0.21) separately, so they are not caused by the origin mix. In natural peptides the hydrophobic fraction also shows a small effect (δ = +0.23).
* Cationic peptides are somewhat more likely to be reported as mammalian-active, but the overlap is large, so charge alone would not predict it well.

## Limitations
* The peptides are not a random sample of all peptides; they are those researchers chose to test and report in DBAASP. Synthetic peptides also outnumber natural ones about 3 to 1 even after redundancy removal.
* CD-HIT keeps the longest sequence of each cluster and merges fragments into longer peptides, so the non-redundant set is slightly biased towards longer peptides. About 20% of monomer sequences were also removed earlier for containing `X` (notebook 01).
* Target groups overlap, so Q2 uses only peptides with exactly one target set (or the Gram+ & Gram- pair). This leaves a fraction of the data and small groups for some targets.
* For Q3, "not reported" is not the same as "not toxic": a peptide may not have been tested on mammalian cells.
* The features are simple sequence-based summaries (no structure, no terminal modifications); the hydrophobic fraction depends on the chosen set of residues.
* These are associations in a curated database, not evidence that a feature *causes* a difference in activity.